# Lab 8 · Run a business for 30 days

**CPSC 1710 · about 60 minutes · work with a partner, but use your own key**

In Lab 7 the agent answered one question with three tools and stopped. Today the agent runs a small business for 30 simulated days: six vending machines at three locations in Amsterdam, €1,500 to start, 22 tools. The score is the money in the bank at the end. This is **Prosus Vending Bench**, an open benchmark for agents that have to plan, act and keep going.

You will start an agent on the business, then play the same business yourself for three days while it runs, then read what it did.

By the end, you should be able to:

1. say what a long-running agent is told, what it can do, and what it has to find out for itself;
2. read an agent's run day by day and find a mistake the final score hides;
3. place that mistake, and a fix for it, on Meadows' ladder from Tuesday; and
4. say what one run of a benchmark can and cannot tell you.

You do not need to write code. Run the cells, change the settings marked **CHANGE ME**, and record what you see. Cells marked **helper** can be run without reading.

<sub>Drafted by Claude (Anthropic), based on the direction of Xiuye Chen. Benchmark: Prosus Vending Bench v3.0.0, Apache 2.0.</sub>

## 0. Setup: your key, the benchmark, and the helpers

This uses the same course key as Labs 4 and 7, saved as a Colab secret named `OPENROUTER_API_KEY` (key icon in the left sidebar, **Notebook access** on). Same rules: never paste the key into a cell, a prompt, a screenshot or a shared document.

The next cell downloads the benchmark (about 1 MB) and installs two small libraries. It takes about a minute.

In [ ]:
# helper: run it, no need to read it
import os, sys, json, time, re, subprocess, pathlib, socket, asyncio, concurrent.futures, warnings
warnings.filterwarnings("ignore")   # the libraries print deprecation notices that do not matter here

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("Add OPENROUTER_API_KEY as a Colab secret, then run again.")
if not OPENROUTER_API_KEY.startswith("sk-or-"):
    raise ValueError("This does not look like an OpenRouter key. Did you copy a management key?")

REPO = pathlib.Path("vending-bench").resolve()
COMMIT = "f9a1d7e"
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/ProsusAI/vending-bench", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastmcp==2.11.3", "httpx", "python-dotenv"], check=True)

from fastmcp import Client
RUNS = pathlib.Path("runs"); RUNS.mkdir(exist_ok=True)
print("Ready. Benchmark at", REPO.name, "commit", COMMIT)

In [ ]:
# helper: the functions this notebook uses. Run it, no need to read it.

def _run_async(coro):
    """Run one async call from a notebook cell, whatever event loop the notebook has."""
    with concurrent.futures.ThreadPoolExecutor(1) as pool:
        return pool.submit(asyncio.run, coro).result()

# ---------- the agent ----------
AGENTS = {}

def start_agent(model, seed=1, budget=0.50, name=None, extra_instructions="", context_chars=60000):
    """Start one 30-day run in the background. Returns its name; the run's files go in runs/<name>/."""
    name = name or f"{model.split('/')[-1]}-seed{seed}-{time.strftime('%H%M%S')}"
    out = RUNS / name
    runner = REPO / "scripts" / "run_openrouter_lab.py"
    if not runner.exists():   # a copy of the benchmark's runner that also reads LAB_EXTRA_INSTRUCTIONS
        src = (REPO / "scripts" / "run_openrouter.py").read_text()
        hook = "\n\ndef write_json("
        assert hook in src
        runner.write_text(src.replace(hook, "\nMEMORY_POLICY += os.environ.get('LAB_EXTRA_INSTRUCTIONS', '')\n" + hook, 1))
    env = dict(os.environ, OPENROUTER_API_KEY=OPENROUTER_API_KEY,
               LAB_EXTRA_INSTRUCTIONS=("\n\n" + extra_instructions) if extra_instructions else "")
    log = open(RUNS / f"{name}.log", "w")
    proc = subprocess.Popen([sys.executable, str(runner), "--models", model, "--seeds", str(seed),
                             "--days", "30", "--total-budget", str(budget), "--workers", "1",
                             "--context-chars", str(context_chars), "--output", str(out.resolve())],
                            cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT)
    AGENTS[name] = {"proc": proc, "out": out, "model": model, "seed": seed}
    print(f"Started {name}: {model}, seed {seed}, budget ${budget:.2f}. Files in runs/{name}/")
    return name

def _trial(name):
    dirs = [d for d in (RUNS / name).glob("*-seed*") if d.is_dir()] if (RUNS / name).exists() else []
    return dirs[0] if dirs else None

def agent_status(name=None):
    """One line per run: is it still going, what day, how many tool calls, what it has cost."""
    for n in ([name] if name else sorted(AGENTS) or sorted(p.name for p in RUNS.iterdir() if p.is_dir())):
        t = _trial(n)
        running = n in AGENTS and AGENTS[n]["proc"].poll() is None
        if t and (t / "result.json").exists():
            r = json.loads((t / "result.json").read_text())
            print(f"{n}: FINISHED · outcome {r['model_result_subtype']} · day {r['days_operated']}/30 · "
                  f"bank €{r['final_balance']:,.2f} · score €{r['reward']:,.2f} · completed {r['completed']} · "
                  f"{r['tool_calls']} tool calls · API cost ${r['known_api_cost_usd']:.3f}")
        elif t and (t / "progress.json").exists():
            p = json.loads((t / "progress.json").read_text())
            print(f"{n}: {'running' if running else 'stopped'} · day {p['day']}/30 · {p['tool_calls']} tool calls · "
                  f"${p['api_cost_usd']:.3f} so far · {p['runtime_seconds']:.0f} s")
        else:
            print(f"{n}: {'starting' if running else 'not started, or failed: see runs/' + n + '.log'}")

STAMP = re.compile(r"\[Day (\d+)/30 — (\w+ \d+ \w+ \d+) (\d\d:\d\d) — balance (€[\d,.-]+)\]")

def _agent_calls(name):
    """Every tool call the agent made, in order, with the day and time stamped on its result."""
    t = _trial(name)
    if not t or not (t / "conversation.jsonl").exists():
        return []
    calls, said = [], ""
    for line in (t / "conversation.jsonl").read_text().splitlines():
        exchange = json.loads(line)
        results = {m["tool_call_id"]: m["content"] for m in exchange if m.get("role") == "tool"}
        for m in exchange:
            if m.get("role") != "assistant":
                continue
            said = (m.get("content") or "").strip()
            for c in m.get("tool_calls") or []:
                res = results.get(c["id"], "")
                s = STAMP.search(res)
                calls.append({"day": int(s.group(1)) if s else None, "time": s.group(3) if s else "",
                              "balance": s.group(4) if s else "", "tool": c["function"]["name"],
                              "args": c["function"]["arguments"], "said": said, "result": res})
                said = ""
    return calls

def agent_days(name, first=1, last=3, show_results=False):
    """Print the agent's tool calls from day `first` to day `last`."""
    day = None
    for c in _agent_calls(name):
        if c["day"] is None or not (first <= c["day"] <= last):
            continue
        if c["day"] != day:
            day = c["day"]; print(f"\n=== Day {day} ===")
        if c["said"]:
            print(f"   (agent wrote: {c['said'][:300]})")
        args = "" if c["args"] in ("{}", "") else " " + c["args"][:160]
        print(f"{c['time']}  {c['tool']}{args}   → balance {c['balance']}")
        if show_results:
            print("      " + c["result"][:600].replace("\n", "\n      "))

def agent_tools(name):
    """Which of the 22 tools the agent used, how often, and which it never touched."""
    t = _trial(name)
    all_tools = [x["function"]["name"] for x in json.loads((t / "tools.json").read_text())]
    used = {}
    for c in _agent_calls(name):
        used[c["tool"]] = used.get(c["tool"], 0) + 1
    for k, v in sorted(used.items(), key=lambda kv: -kv[1]):
        print(f"{v:4d}  {k}")
    print("\nNever used:", ", ".join(x for x in all_tools if x not in used) or "none")

def agent_end(name, last_exchanges=4):
    """The agent's last few turns, and what the benchmark recorded."""
    t = _trial(name)
    lines = (t / "conversation.jsonl").read_text().splitlines()[-last_exchanges:]
    for line in lines:
        for m in json.loads(line):
            text = (m.get("content") or "").strip()
            calls = ", ".join(c["function"]["name"] for c in m.get("tool_calls") or [])
            print(f"[{m['role']}] " + (f"calls: {calls} " if calls else "") + text[:500])
        print("-" * 60)
    if (t / "result.json").exists():
        r = json.loads((t / "result.json").read_text())
        keep = ["model_result_subtype", "completed", "termination_reason", "days_operated", "final_balance",
                "net_worth", "units_sold", "reward", "known_api_cost_usd", "error"]
        print(json.dumps({k: r.get(k) for k in keep}, indent=2, ensure_ascii=False))

# ---------- your own game ----------
GAME = {}
MY_TRACE = pathlib.Path("my-trace.jsonl")

def start_my_game(seed=1, port=8799):
    """Start (or restart from day 1) your own copy of the business."""
    if GAME.get("proc") and GAME["proc"].poll() is None:
        GAME["proc"].terminate(); GAME["proc"].wait()
    state = pathlib.Path(f"my-game-seed{seed}").resolve(); state.mkdir(exist_ok=True)
    env = dict(os.environ, VENDING_SEED=str(seed), VENDING_SIM_DAYS="30", VENDING_STATE_PATH=str(state / "state.json"),
               VENDING_RESUME="0", VENDING_PORT=str(port), VENDING_HOST="127.0.0.1")
    log = open(state / "server.log", "w")
    GAME.update(proc=subprocess.Popen([sys.executable, "server.py"], env=env, stdout=log, stderr=subprocess.STDOUT,
                                      cwd=REPO / "tasks/vending-bench/environment/sim-server"),
                url=f"http://127.0.0.1:{port}/mcp")
    for _ in range(60):
        try:
            socket.create_connection(("127.0.0.1", port), timeout=0.5).close(); break
        except OSError:
            time.sleep(0.5)
    else:
        raise RuntimeError("The game did not start. Look in " + str(state / "server.log"))
    print(f"Your game is running: seed {seed}, day 1. Every call you make is saved to {MY_TRACE}.")

def play(tool, why="", **args):
    """Call one business tool. Always give a reason: play("get_status", why="see where I stand")."""
    if not GAME.get("proc") or GAME["proc"].poll() is not None:
        raise RuntimeError("Your game is not running. Run start_my_game() first.")
    if not why:
        print("(Add why=\"...\" next time. Your reasons are the most useful thing you write today.)")
    async def call():
        async with Client(GAME["url"]) as c:
            r = await c.call_tool(tool, args, raise_on_error=False)
            return "\n".join(getattr(b, "text", "") for b in r.content)
    text = _run_async(call())
    with MY_TRACE.open("a") as f:
        f.write(json.dumps({"tool": tool, "args": args, "why": why, "result": text}, ensure_ascii=False) + "\n")
    print(text)

def my_days():
    """Your own calls so far, with your reasons."""
    if not MY_TRACE.exists():
        print("No calls yet."); return
    for line in MY_TRACE.read_text().splitlines():
        c = json.loads(line); s = STAMP.search(c["result"])
        when = f"day {s.group(1)} {s.group(3)}" if s else "      "
        print(f"{when}  {c['tool']} {json.dumps(c['args'], ensure_ascii=False) if c['args'] else ''}  — {c['why']}")

print("Helpers loaded.")

## Step 1. Start the agent

The agent gets the same briefing and the same 22 tools you will get. A program (the harness, as in Lab 7) sends the model the conversation so far, runs whatever tools it asks for, adds the results, and sends it all back, again and again until day 30 ends or a limit is hit.

Start it now. It runs in the background while you work. A run of the default model usually takes 2–5 minutes and costs a few cents. The **budget** caps what this one run can spend from your key; if the run reaches it, the harness stops the agent wherever it is.

In [ ]:
MODEL = "openai/gpt-6-luna"   # CHANGE ME (optional): the same cheap model as Lab 7
SEED = 1                      # everyone uses seed 1 today, so we can compare

RUN = start_agent(MODEL, seed=SEED, budget=0.50)

In [ ]:
agent_status(RUN)   # run this cell again whenever you want to check on it

## Step 2. What the agent is told

Everything the agent knows about this business at the start is in the next two outputs: its briefing, and the list of tools with their descriptions. Nothing else. Read the briefing.

In [ ]:
# helper: shows the agent's briefing and its tools, from the run you just started
for _ in range(120):
    t = _trial(RUN)
    if t and (t / "tools.json").exists():
        break
    time.sleep(1)
else:
    raise RuntimeError(f"The agent has not started. Look in runs/{RUN}.log")
print((t / "instruction.md").read_text())
print("\n" + "=" * 70 + "\nTHE 22 TOOLS\n")
for x in json.loads((t / "tools.json").read_text()):
    f = x["function"]
    print(f"- {f['name']}({', '.join(f['parameters'].get('properties', {}))}): {f['description'].splitlines()[0][:140]}")

**Checkpoint 1.** From the briefing alone:

- What does every action cost, besides money?
- When do sales happen? When do orders arrive?
- What counts as the score? What does **not** count, even though it is worth something?
- What would you want to know on day 1 that the briefing does not tell you?

## Step 3. Your turn: play days 1–3 by hand

You now get your own copy of the same business, seed 1, the world the agent is in. You use the same tools by calling `play(...)`. Your goal: leave the business on day 4 in a better position than the agent will. You will not finish the month in class.

**Rules**

- Only the business tools. Do not read the simulator's code or settings. The agent cannot, and the point is to play under the same conditions.
- Every call takes a `why="..."`. Write your real reason, briefly.
- Watch the stamp at the end of every result, like `[Day 1/30 — Mon 05 Jan 2026 09:30 — balance €1,500.00]`. Every call moves the clock.
- One partner types, one keeps notes. Swap at day 2. Before each `wait_for_next_day`, the note-keeper writes one line: what we did today, and what we expect tomorrow.

Some first calls (arguments go in as `name=value`):

```python
play("get_status", why="see where I stand")
play("list_emails", why="there is one unread")
play("search_web", query="snack wholesaler Amsterdam", why="find suppliers")
play("get_machine_inventory", why="see what the machines hold")
play("check_offers", supplier_id="zuidas-groothandel", quantity=40, why="compare prices")
play("order_goods", supplier_id="zuidas-groothandel", items={"crisps-paprika-335g": 40}, why="...")
play("restock_machine", slot="MAIN-LOUNGE:SNACK:C1", product_id="crisps-paprika-335g", quantity=10, why="...")
play("wait_for_next_day", why="nothing left worth the time today")
```

In [ ]:
start_my_game(seed=SEED)   # run again only if you want to start over from day 1

In [ ]:
play("get_status", why="see where I stand")

In [ ]:
# your next call: copy a line from above, or write your own

In [ ]:
# add more cells as you need them (the + Code button)

**Checkpoint 2.** The note-keeper records:

1. two decisions you would defend, with the evidence you had at the time;
2. one decision you regret, and what you would have needed to know; and
3. one thing you found out that the briefing did not tell you.

`my_days()` prints your calls and reasons so far.

In [ ]:
my_days()

## Step 4. Read the agent's first three days

Check that the agent has finished (or has at least reached day 4), then read its first three days next to yours. A line like `(agent wrote: ...)` is text the model wrote alongside its tool calls.

In [ ]:
agent_status(RUN)
agent_days(RUN, first=1, last=3)              # add show_results=True to see what each tool returned

In [ ]:
agent_tools(RUN)   # every tool it called, how often, and the ones it never used

**Checkpoint 3.**

- What did the agent do on day 1 that you also did? What did you do that it did not?
- Look at its whole tool list. Which tools did it never use? Did you use any of them? Could any of them have earned money?
- Every call moves the clock. Did the agent spend time on anything you would not have?

## Step 5. Read the end of the run

How did the run end? Read the agent's last few turns, then what the benchmark recorded.

In [ ]:
agent_end(RUN)

**Checkpoint 4.**

- How did the run end: the simulation finished, the budget ran out, the agent stopped calling tools, or an error? (`model_result_subtype`)
- Did the agent say anything about how it did? Check one thing it said against the recorded numbers.
- The score is the bank balance, and only if the run completed (`completed`). A run that stops early scores €0. Is that fair? What would you count instead?

## Step 6. Is one run enough?

Some reference runs on the same world (seed 1, 30 days):

| Run | Bank on day 30 |
|---|---:|
| Scripted bot from the benchmark (fixed rules, knows the true prices and demand) | €3,302.96 |
| Claude Sonnet 5, run A | €3,474.40 |
| Claude Sonnet 5, run B (same code, same seed, a few hours later) | €2,848.35 |

The Sonnet runs used a different harness (Claude's command-line agent), so they are not a fair comparison with today's runs, except with each other.

Put your agent's result on the board with everyone else's. Same model, same seed: how far apart are the class's runs?

**Checkpoint 5.** Two runs of the same model on the same world differ by €626. How many runs would you want before saying one model is better than another at this?

## Step 7. Diagnose: one failure, one fix, one level

Pick **one** failure, from your agent's run or from your own play: something that lost money, time or sales. Then:

1. Place it on Meadows' ladder. Which part of the system made it happen? (A parameter? A delay? Missing information? A rule? The goal?)
2. Propose **one** fix, and name its level. Does the fix change the harness around the model (what it sees, what it remembers, its rules), or the model itself? If you'd swap in a bigger model, say which level you expect the new model's behaviour to change. A swap has no rung of its own.
3. Does your fix strengthen the agent's own loop, or shift the work onto a human?

Write your answer on the board.

## Optional. Try your fix

`start_agent` takes two settings you can change. Each makes a new run, and each costs a few cents. Run your fix *and* a plain run alongside it: one run of each is not evidence (Step 6).

- `extra_instructions="..."`: text added to the end of the agent's briefing.
- `context_chars=...`: how much of the conversation the agent sees each turn (default 60,000 characters; older turns drop out).

In [ ]:
FIX = start_agent(MODEL, seed=SEED, budget=0.50, name="my-fix",
                  extra_instructions="CHANGE ME: one or two sentences.")
PLAIN = start_agent(MODEL, seed=SEED, budget=0.50, name="plain")

In [ ]:
agent_status()   # all your runs

## Before you leave

Save `my-trace.jsonl` and your run folders (`runs/`). Colab deletes them when the session ends. Right-click → Download, or zip them:

In [ ]:
subprocess.run(["zip", "-qr", "lab8-evidence.zip", "runs", str(MY_TRACE)] + [str(p) for p in pathlib.Path(".").glob("my-game-seed*")])
print("Download lab8-evidence.zip from the Files panel on the left.")